Notebook to inspect the steps of the Neural Slicer code

In [30]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pyvista as pv
import torch
import trimesh
from utils.fileIO import loadTet
from utils.pv_tetIO import loadTet as pv_loadTet
from deformationOptimization import deformationOptimization

pv.start_xvfb()
# Change to "html" for dynamic plots in Jupyter - if using html, on each cloud instance boot-up, you currently need to install the `jupyter-widgets` extension from the JupyterLab extension tab in this window and refresh the page.
pv.set_jupyter_backend('html')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
EXPERIMENT_NAME = "spiral_fish"

original_geometry_dir = Path("./data/TET_MODEL")
# Find the file in the original geometry directory with the same name as the experiment
original_geometry_file = next(original_geometry_dir.glob(f"{EXPERIMENT_NAME}*"))

all_results_dir =  Path("./data/results") / EXPERIMENT_NAME
# Find the latest experiment directory
latest_experiment_dir = max(all_results_dir.iterdir(), key=lambda x: x.stat().st_mtime)
# Get path of highest numbered heightField_[number].txt file
latest_height_field_file = max(latest_experiment_dir.glob("heightField_*.txt"), key=lambda x: x.stat().st_mtime)
optimised_model_file = latest_experiment_dir / "last.ckpt"
optimised_model_latent_file = latest_experiment_dir / "latent.pt"

deformed_surface_mesh = pv.read(latest_experiment_dir / "outCage.obj")

print(f"Original geometry path: {original_geometry_file}")
print(f"Pulling results from: {latest_experiment_dir}\n")

Original geometry path: data/TET_MODEL/spiral_fish.tet
Pulling results from: data/results/spiral_fish/2025_04_14-08_04_33



### Step 0: Original Geometry

In [79]:
pv_original_mesh = pv_loadTet(original_geometry_file)
pv_original_mesh.plot()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

### Step 1: Cage-based Field Generation

See results from running `main.py`

In [57]:
original_tetmesh = loadTet(original_geometry_file)

# Generate deformed surface mesh through model
wrapper = torch.load(optimised_model_file, weights_only=False)
latent = torch.load(optimised_model_latent_file)
wrapper.eval()

extra_args = {'nstep': 5000, 'resume_train': False, 'mesh_name': 'spiral_fish.tet', 'cage_name': 'None', 'cage_face_num': None, 'stress_name': 'None', 'exp_name': 'spiral_fish', 'id': '2025_04_14-10_10_21', 'model': '', 'lrate': 0.0001, 'min_lr': 1e-06, 'factor': 0.9, 'patience': 20, 'cooldown': 200, 'optimizer': 'adam', 'wSF': 1.0, 'wSR': 0.0, 'wSQ': 0.0, 'wOP': 0.0, 'wConstraints': 5.0, 'wSF_Lattice': 0.0, 'wSR_Lattice': 0.0, 'wSF_Shell': 0.0, 'wSR_Shell': 0.0, 'wSF_Tube': 0.0, 'wSR_Tube': 0.0, 'wRigid': 100.0, 'wScaling': 10.0, 'wQuaternion': 0.01, 'wRegulation': 0.0001, 'wRegulation1': 10000.0, 'alpha': 45, 'beta': 2.0, 'grammar': 5, 'lock_bottom': True, 'mesh_dir': './data/TET_MODEL/', 'cage_dir': './data/cage/', 'stress_dir': './data/fem_result/', 'result_dir': './data/results/', 'use_comet': False}

do = deformationOptimization(mesh=original_tetmesh, cage=None, stress=None, latent=latent, wrapper=wrapper, **extra_args)

In [87]:
# Run one dummy training step to generate optMeshBoundaryNodes
do.arap.loss(do.wrapper(do.arap.elemCenter, latent=latent), step = 0)

pv_deformed_mesh = pv_original_mesh.copy()
pv_deformed_mesh.points = do.arap.optMeshNodes.squeeze(0).detach().numpy()

print(f"{pv_original_mesh.n_points} points in original mesh")
print(f"{pv_deformed_mesh.n_points} points in deformed mesh")

pl = pv.Plotter()

pl.add_mesh(pv_original_mesh, name="Original Mesh", opacity=0.25, color="blue")
pl.add_mesh(pv_deformed_mesh, name="Deformed Mesh", opacity=1.0, color="red")
pl.show()

11687 points in original mesh
11687 points in deformed mesh


EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…